# 🚀 DeepScalper Mission Control Plane
**Project:** FinRL-Pro_DS | **Architecture:** Mach 3 (RTX 5090)

This notebook serves as the interactive **Cockpit** for the MLOps pipeline. It consolidates all CLI utilities into a visual workflow using the scripts in `scripts/`.

### 📋 Workflow Stages:
1.  **Environment Setup**: Verify Python paths.
2.  **Configuration**: Review/Edit `configs/deepscalper_unified.yaml`.
3.  **Mission Launch**: Deploy code to GPUHub/RunPod and start execution.
4.  **Monitoring**: Real-time checking of remote processes.
5.  **Analysis**: Fetch results and log findings.

## 1. Environment Setup
**Goal:** Ensure the notebook kernel sees the `finrl_pro_ds` package.
**Action:** Adds the project root to `sys.path` and changes the working directory.

In [ ]:
import os
import sys
import yaml
import subprocess
import pandas as pd
from datetime import datetime
import time

# Ensure Project Root is in Path
PROJECT_ROOT = os.path.abspath("../")
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)
    
os.chdir(PROJECT_ROOT)
print(f"✅ Working Directory set to: {os.getcwd()}")

## 2. Configuration Management
**Goal:** Define the exact parameters for this mission.
**File:** `configs/deepscalper_unified.yaml`

This cell loads the master YAML config which controls:
- **Network**: Model architecture dimensions (Input, Hidden, Output)
- **Env**: Number of parallel environments (`num_envs`) and shared memory settings.
- **Training**: Total timesteps, batch size, and learning rates.

In [ ]:
# Load current config
CONFIG_PATH = "configs/deepscalper_unified.yaml"
with open(CONFIG_PATH, "r") as f:
    config = yaml.safe_load(f)

print(f"🔹 Current Architecture: {config['network']['hidden_size']} units")
print(f"🔹 Current Agents: {list(config['agents'].keys())}")
print(f"🔹 Training timesteps: {config['training']['total_timesteps']}")

In [ ]:
# 🛠️ Quick Edit: Run this cell to update Training Steps or Batch Size temporarily
config['training']['total_timesteps'] = 5000000
config['training']['batch_size'] = 4096

# Save back (Optional - Uncomment to overwrite)
# with open(CONFIG_PATH, "w") as f:
#     yaml.dump(config, f)
# print("✅ Configuration Updated!")

## 3. Mission Launch (Deployment)
**Goal:** Package code, upload dependencies, and start execution on the remote server.
**Script:** `scripts/deploy_bare_metal.py`

### Mission Types:
- **`pipeline`**: Runs `scripts/run_full_pipeline.py`. Sequentially executes:
  1. **Training** (`scripts/train_deepscalper_v3.py`)
  2. **Backtesting** (`scripts/backtest_deepscalper.py`) - Saves metrics & plots.
  3. **Reporting** (`scripts/generate_report.py`) - Generates Audit Report.
- **`hpo`**: Runs `scripts/tune_deepscalper.py`. Executes Hyperparameter Optimization sweep.
- **`train`**: Runs `scripts/train_deepscalper_v3.py`. Executes a single training run.

In [ ]:
# 🎛️ Mission Parameters
MISSION_TYPE = "pipeline" # 'train', 'hpo', or 'pipeline' (train -> backtest)
TARGET = "gpuhub"    # 'gpuhub' or 'runpod'
RUN_ID = f"DS_Mission_{datetime.now().strftime('%Y%m%d_%H%M')}"
NOTES = "Alpha HPO Sweep on Full Dataset"

HOST = "root@<LAN_HOST>" # REPLACE WITH ACTUAL HOST
KEY_PATH = "~/.ssh/id_rsa"  # REPLACE WITH ACTUAL KEY PATH

print(f"🚀 Preparing Mission: {RUN_ID}")
print(f"   Target: {TARGET} ({HOST})")

In [ ]:
# 🟢 EXECUTE LAUNCH
# This runs the bare_metal deployer script which:
# 1. Zips the current codebase (excluding big data/logs)
# 2. Uploads via SFTP
# 3. Installs dependencies from requirement.txt
# 4. Launches the selected script via 'nohup'

# Select Script based on Mission Type
if MISSION_TYPE == "hpo":
    SCRIPT = "scripts/tune_deepscalper.py"
elif MISSION_TYPE == "train":
    SCRIPT = "scripts/train_deepscalper_v3.py"
elif MISSION_TYPE == "pipeline":
    SCRIPT = "scripts/run_full_pipeline.py"
else:
    raise ValueError(f"Unknown Mission Type: {MISSION_TYPE}")

cmd = [
    "python", "scripts/deploy_bare_metal.py",
    "--host", HOST,
    "--key", KEY_PATH,
    "--run_name", RUN_ID,
    "--script", SCRIPT
]

print(f"Running: {' '.join(cmd)}")
# subprocess.run(cmd) # Uncomment to Run

## 4. In-Flight Monitoring
**Goal:** Verify the mission is running and healthy without SSH-ing in manually.
**Scripts:**
- `scripts/monitor_status.py`: Checks if `python` process is alive.
- `scripts/remote_cmd.py`: Executes shell commands (like `tail`) remotely.

In [ ]:
# Check Process Status
!python scripts/monitor_status.py --host {HOST} --key {KEY_PATH}

In [ ]:
# Stream Logs (Tail)
# View the last 50 lines of the run log to check for errors or progress
!python scripts/remote_cmd.py "tail -n 50 /workspace/DeepScalper/run.log" --host {HOST} --key {KEY_PATH}

## 5. Post-Flight Analysis & Logging
**Goal:** Download critical artifacts and document the run.
**Scripts:**
- `scripts/fetch_db.py` (or `fetch_results.py`): Downloads logs, checkpoints, and reports.
  - *Note: This will now fetch the Audit Report generated by Phase 3.*
- `scripts/log_research.py`: Appends entry to `randd_log.md` using the Research Logger skill.

In [ ]:
# Fetch Results
# This will verify if 'results/' folder exists and download it.
!python scripts/fetch_db.py --host {HOST} --key {KEY_PATH}
print("✅ Data retrieved to local directory.")

In [ ]:
# Auto-Log to Research Journal (randd_log.md)
# Automatically updates your Markdown flight log.

log_cmd = [
    "python", ".agent/skills/research_logger/scripts/log_research.py",
    "--title", f"Mission Report: {RUN_ID}",
    "--objective", NOTES,
    "--issue", "Pending Analysis",
    "--solution", "Executed Standard Pipeline",
    "--conclusion", "Pending Verification"
]

print(f"Logging: {' '.join(log_cmd)}")
# subprocess.run(log_cmd) # Uncomment to Log